# Part 3: Framework 2 – TruLens (Feedback Functions & TruVis) & Automated CI/CD Testing
While frameworks like Ragas are built primarily for static, batch dataset evaluations (e.g., scoring 500 test questions in a notebook or job), TruLens takes an observability and instrumentation-first approach.

TruLens wraps your live application code using Feedback Functions to trace every execution, log records in real time, and integrate evaluation directly into automated CI/CD pipelines as a deployment quality gate.

## 1. Core Architecture: TruLens Feedback Functions
TruLens evaluates applications using Feedback Functions, which are modular, reusable Python callables designed to score a specific behavior or attribute of a trace record.

The Three Core Feedback Functions (The TruLens RAG Triad Implementation)
Context Relevance (f_context_relevance): Measures how relevant the retrieved chunks are to the user's input query.

Groundedness (f_groundedness): Measures whether the generated answer is supported by the retrieved context chunks (hallucination detector).

Answer Relevance (f_answer_relevance): Measures whether the final response addresses the user's initial question.

## 2. Implementation Pattern: Instrumenting a RAG App with TruLens
Here is how you wrap a functional RAG query pipeline with TruLens feedback functions and recorder objects:

In [ ]:
"""
trulens_rag_evaluation.py
Demonstrates instrumenting a RAG pipeline with TruLens feedback functions
to evaluate Context Relevance, Groundedness, and Answer Relevance dynamically.
"""

import os
from trulens.core import TruSession
from trulens.apps.langchain import TruChain
from trulens.providers.openai import OpenAI as TruOpenAI
from trulens.core import Feedback
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def build_instrumented_rag_app():
    # 1. Setup a simple Chroma vector store with mock documents
    embeddings = OpenAIEmbeddings()
    vectorstore = Chroma.from_texts(
        texts=[
            "TechCorp Europe experienced supply chain bottlenecks in Q3 due to port strikes in Berlin.",
            "DataStream Logistics provides tier-1 automated routing solutions across Germany."
        ],
        embedding=embeddings
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

    # 2. Setup standard LLM and prompt chain
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer the question strictly using the provided context:\n\n{context}"),
        ("human", "{input}")
    ])

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # Build LangChain LCEL RAG chain
    rag_chain = (
        {"context": retriever | format_docs, "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain, retriever

def setup_trulens_feedbacks():
    """Defines the TruLens evaluation feedback functions using OpenAI as a judge."""
    provider = TruOpenAI()

    # Feedback 1: Context Relevance (Query vs Retrieved Context)
    f_context_relevance = (
        Feedback(provider.qs_relevance, name="Context Relevance")
        .on_input()
        .on(TruChain.select_retriever().rets)
    )

    # Feedback 2: Groundedness (Retrieved Context vs Generated Answer)
    f_groundedness = (
        Feedback(provider.groundedness_of_response, name="Groundedness")
        .on(TruChain.select_retriever().rets)
        .on_output()
    )

    # Feedback 3: Answer Relevance (Query vs Generated Answer)
    f_answer_relevance = (
        Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
        .on_input()
        .on_output()
    )

    return [f_context_relevance, f_groundedness, f_answer_relevance]

if __name__ == "__main__":
    if "OPENAI_API_KEY" not in os.environ:
        print("Error: OPENAI_API_KEY environment variable is required.")
    else:
        # Initialize TruLens Session (stores evaluation data locally or in sqlite)
        session = TruSession()
        session.reset_database()

        # Build app and feedbacks
        chain, retriever = build_instrumented_rag_app()
        feedbacks = setup_trulens_feedbacks()

        # Wrap app with TruChain recorder
        tru_recorder = TruChain(
            chain,
            app_id="EnterpriseRAG_V1",
            feedbacks=feedbacks
        )

        # Execute queries through the recorder
        with tru_recorder as recording:
            query = "Why did TechCorp Europe face supply chain issues?"
            response = chain.invoke(query)
            print(f"\nUser Query: {query}")
            print(f"Generated Response: {response}")

        # View aggregated metrics or launch dashboard
        print("\nTruLens evaluation recorded successfully.")
        records, feedback_results = session.get_records_and_feedback(app_ids=["EnterpriseRAG_V1"])
        print(records[["input", "output", "Groundedness", "Context Relevance", "Answer Relevance"]])

## 3. Integrating Evaluation into Automated CI/CD Testing
To prevent regressions when changing prompt templates, chunk sizes, or embedding models, evaluation should run automatically inside your CI/CD pipeline (e.g., GitHub Actions).

CI/CD Quality Gate Pattern (test_rag_quality_gate.py)
You can write a standard pytest file that asserts minimum performance thresholds before allowing a code merge

In [ ]:
"""
test_rag_quality_gate.py
Automated test suite asserting minimum RAG triad scores for CI/CD deployment gates.
"""

import pytest
from ragas import evaluate
from datasets import Dataset
from ragas.metrics import Faithfulness, ResponseRelevancy
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

@pytest.fixture
def evaluated_rag_metrics():
    # Simulate running your golden test set against your RAG pipeline
    test_dataset = Dataset.from_dict({
        "user_input": ["What caused the Berlin supply chain issues?"],
        "retrieved_contexts": [["TechCorp Europe experienced supply chain bottlenecks in Q3 due to port strikes in Berlin."]],
        "response": ["Port strikes in Berlin caused the supply chain bottlenecks for TechCorp Europe."]
    })
    
    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
    metrics = [Faithfulness(llm=evaluator_llm), ResponseRelevancy(llm=evaluator_llm)]
    
    results = evaluate(dataset=test_dataset, metrics=metrics)
    return results.to_pandas()

def test_faithfulness_threshold(evaluated_rag_metrics):
    """Ensures that the model's faithfulness score meets enterprise deployment standards."""
    mean_faithfulness = evaluated_rag_metrics["faithfulness"].mean()
    print(f"Measured Faithfulness Score: {mean_faithfulness}")
    
    # Assert that responses are at least 85% faithful to the source material
    assert mean_faithfulness >= 0.85, f"Deployment blocked: Faithfulness score {mean_faithfulness} fell below threshold 0.85"

def test_relevancy_threshold(evaluated_rag_metrics):
    """Ensures that responses directly address user intent."""
    mean_relevancy = evaluated_rag_metrics["response_relevancy"].mean()
    print(f"Measured Response Relevancy Score: {mean_relevancy}")
    
    assert mean_relevancy >= 0.85, f"Deployment blocked: Relevancy score {mean_relevancy} fell below threshold 0.85"